In [34]:
import os
from openai import AzureOpenAI
from tqdm import tqdm
import time
import re
from sklearn.metrics import *
import krippendorff
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import *

In [35]:
!nvidia-smi

Tue Mar 10 03:39:42 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.247.01             Driver Version: 535.247.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100 80GB PCIe          On  | 00000001:00:00.0 Off |                    0 |
| N/A   31C    P0              42W / 300W |      0MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [36]:
api_key = os.getenv("AZURE_OPENAI_API_KEY_MODEL")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT_MODEL")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_MODEL")
api_version = os.getenv("AZURE_OPENAI_API_VERSION_MODEL")

In [37]:
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=api_key,
    api_version=api_version,
)

In [38]:
response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

In [39]:
print(response.choices[0].message.content)

Paris.


In [40]:
# import os
# from openai import AzureOpenAI

# endpoint = ""
# model_name = "gpt-5.2-chat"
# deployment_name = "gpt-5.2-chat"

# subscription_key = ""
# api_version = "2024-12-01-preview"

# client = AzureOpenAI(
#     api_version=api_version,
#     azure_endpoint=endpoint,
#     api_key=subscription_key,
# )

# response = client.chat.completions.create(
#     messages=[
#         {
#             "role": "system",
#             "content": "You are a helpful assistant.",
#         },
#         {
#             "role": "user",
#             "content": "I am going to Paris, what should I see?",
#         }
#     ],
#     max_completion_tokens =16384,
#     model=deployment_name
# )

# print(response.choices[0].message.content)

In [41]:
def query_llm(sys_msg, query_msg, dt=1e-2):
    time.sleep(dt)
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": query_msg}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(e)
        return -1

# Load Emotions Dataset

In [42]:
df = pd.read_csv("emotion_data/full_dataset/test_df.csv")

In [43]:
class_cols = ['admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral']
class_cols = np.array(class_cols)

In [44]:
df['label'] = class_cols[df[class_cols].to_numpy().argmax(-1)]
df['label_encoded'] = df[class_cols].to_numpy().argmax(-1)

In [45]:
df['label']

0              neutral
1              neutral
2           admiration
3       disappointment
4           admiration
             ...      
1013         gratitude
1014          surprise
1015         amusement
1016           neutral
1017        admiration
Name: label, Length: 1018, dtype: object

In [46]:
df.columns

Index(['Unnamed: 0', 'index', 'text', 'id', 'author', 'subreddit', 'link_id',
       'parent_id', 'created_utc', 'rater_id', 'example_very_unclear',
       'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
       'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
       'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
       'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
       'relief', 'remorse', 'sadness', 'surprise', 'neutral', 'label',
       'label_encoded'],
      dtype='object')

In [47]:
df[['text', 'label']]

,text,label
0,"Dirt, glass, poop whatever you dont want on yo...",neutral
1,[NAME] has personal experience with people tur...,neutral
2,These look fantastic. Please go and work for t...,admiration
3,I sit on one of my legs. I always have my othe...,disappointment
4,"the lack of respect for [NAME], [NAME] and the...",admiration
...,...,...
1013,"It was [NAME] thank you very much, than I'd go...",gratitude
1014,Between this and the headline it makes me wond...,surprise
1015,asking for a source is good lol,amusement
1016,Shoulda told her to come watch it at ur place,neutral


# LLM Prompt

In [48]:
system_prompt = """You will be given a text, and your task is to classify it into a sentiment.
This is the list of sentiments you can choose from:
admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief, joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise, neutral.

Just output one of the emotions. There is no need to provide any explanation.
"""

query = """<Start of text>
{}
<End of text>

The type of sentiment the text display is: """

In [49]:
def get_llm_prompt(text):
    user_query = query.format(text)
    return system_prompt, user_query

In [50]:
idx = 9

In [51]:
sys_prompt, query_prompt = get_llm_prompt(df['text'][idx])

In [52]:
df['text'][idx]

'How long do we have to wait? I might need rehab.'

In [53]:
df['label'][idx]

'curiosity'

In [54]:
query_llm(sys_prompt, query_prompt)

'nervousness'

# LLM Scoring

In [55]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
from functools import partial
from tqdm import tqdm
import time

# reuse your query_llm as-is (remove the test sleep dt)
# def query_llm(sys_msg, query_msg, dt=1e-2): ...

def parse_response_text(text):
    if text in class_cols:
        return text
    else:
        return "neutral"

async def run_queries_async(df, max_workers=10, retries=2):
    """
    df: dataframe like your df
    max_workers: number of threads to run in parallel (tuneable)
    retries: per-item retry attempts for transient errors
    """
    loop = asyncio.get_running_loop()
    executor = ThreadPoolExecutor(max_workers=max_workers)
    results = [None] * df.shape[0]
    tasks = []

    async def worker(i):
        sys_msg, q_msg = get_llm_prompt(df['text'][i])
        attempt = 0
        while True:
            attempt += 1
            try:
                # run the blocking query in a thread
                func = partial(query_llm, sys_msg, q_msg)
                text = await loop.run_in_executor(executor, func)
                results[i] = parse_response_text(text)
                return  # success
            except Exception as e:
                # query_llm already prints e and returns -1 on exception,
                # but we guard here in case other exceptions arise.
                if attempt > retries:
                    results[i] = "neutral"
                    return
                # small backoff (blocking sleep happens in thread; here we await)
                await asyncio.sleep(0.5 * attempt)

    # create tasks
    for i in range(df.shape[0]):
        tasks.append(asyncio.create_task(worker(i)))

    # update progress as tasks finish
    with tqdm(total=len(tasks)) as pbar:
        for finished in asyncio.as_completed(tasks):
            await finished
            pbar.update(1)

    executor.shutdown(wait=True)
    return results

In [56]:
# Tune max_workers — start with 5-20 depending on API limits/network.
pred_emotions = await run_queries_async(df, max_workers=10, retries=2)
pred_emotions = np.array(pred_emotions)

  3%|▎         | 27/1018 [00:10<05:19,  3.10it/s]

Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': True, 'severity': 'medium'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


 10%|▉         | 97/1018 [00:32<03:23,  4.54it/s]

Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': True, 'severity': 'medium'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


 15%|█▌        | 153/1018 [00:50<04:53,  2.95it/s]

...

 78%|███████▊  | 793/1018 [04:24<01:15,  2.97it/s]

Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': True, 'severity': 'medium'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


 99%|█████████▉| 1011/1018 [05:29<00:01,  4.23it/s]

Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': True, 'severity': 'high'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


100%|██████████| 1018/1018 [05:42<00:00,  2.98it/s]


In [57]:
pred_emotions

array(['disgust', 'neutral', 'admiration', ..., 'approval', 'remorse',
       'approval'], shape=(1018,), dtype='<U14')

In [58]:
(df['label'] == pred_emotions).mean()

np.float64(0.5137524557956779)

In [59]:
f1_score(df['label'], pred_emotions, average='macro')

0.38403780302070223

In [60]:
np.save(f"emotion_data/llm_pred/{deployment_name}_run_3.npy", pred_emotions)

In [61]:
deployment_name

'gpt-5-nano'

# Score using Saved Arrays

In [15]:
!cd emotion_data/llm_pred && ls

gpt-4o_run_1.npy		 gpt-5-nano_run_1.npy
gpt-4o_run_2.npy		 gpt-5-nano_run_2.npy
gpt-4o_run_3.npy		 gpt-5-nano_run_3.npy
gpt-5-mini-2025-08-07_run_1.npy  gpt-5.2-chat_run_1.npy
gpt-5-mini-2025-08-07_run_2.npy  gpt-5.2-chat_run_2.npy
gpt-5-mini-2025-08-07_run_3.npy  gpt-5.2-chat_run_3.npy


In [17]:
def macro_f1_score(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

In [86]:
def compute_scores(filenames, metrics, label):
    all_pred = []
    for file in filenames:
        file = f"emotion_data/llm_pred/{file}"
        arr = np.load(file)
        all_pred.append(arr)
    
    all_scores = np.zeros((len(all_pred), len(metrics)))
    for i in range(len(all_pred)):
        for j in range(len(metrics)):
            pred = all_pred[i]
            metric = metrics[j]
            all_scores[i, j] = metric(label, pred)
    all_scores = pd.DataFrame(all_scores, index=filenames, columns=[m.__name__ for m in metrics])
    all_scores.loc['standard deviation'] = [np.std(all_scores.iloc[:, i]) for i in range(len(metrics))]
    all_scores.loc['average'] = [np.mean(all_scores.iloc[:-1, i]) for i in range(len(metrics))]
    return all_scores

In [87]:
compute_scores(
    ["gpt-4o_run_1.npy", "gpt-4o_run_2.npy", "gpt-4o_run_3.npy"],
    [accuracy_score, macro_f1_score],
    df['label']
)

,accuracy_score,macro_f1_score
gpt-4o_run_1.npy,0.471513,0.371834
gpt-4o_run_2.npy,0.479371,0.376040
gpt-4o_run_3.npy,0.471513,0.371903
standard deviation,0.003705,0.001967
average,0.474132,0.373259


In [88]:
compute_scores(
    ["gpt-5-mini-2025-08-07_run_1.npy", "gpt-5-mini-2025-08-07_run_2.npy", "gpt-5-mini-2025-08-07_run_3.npy"],
    [accuracy_score, macro_f1_score],
    df['label']
)

,accuracy_score,macro_f1_score
gpt-5-mini-2025-08-07_run_1.npy,0.471513,0.382869
gpt-5-mini-2025-08-07_run_2.npy,0.473477,0.393285
gpt-5-mini-2025-08-07_run_3.npy,0.474460,0.386422
standard deviation,0.001225,0.004323
average,0.473150,0.387525


In [89]:
compute_scores(
    ["gpt-5-nano_run_1.npy", "gpt-5-nano_run_2.npy", "gpt-5-nano_run_3.npy"],
    [accuracy_score, macro_f1_score],
    df['label']
)

,accuracy_score,macro_f1_score
gpt-5-nano_run_1.npy,0.517682,0.407730
gpt-5-nano_run_2.npy,0.521611,0.399072
gpt-5-nano_run_3.npy,0.513752,0.384038
standard deviation,0.003208,0.009788
average,0.517682,0.396946


In [90]:
compute_scores(
    ["gpt-5.2-chat_run_1.npy", "gpt-5.2-chat_run_2.npy", "gpt-5.2-chat_run_3.npy"],
    [accuracy_score, macro_f1_score],
    df['label']
)

,accuracy_score,macro_f1_score
gpt-5.2-chat_run_1.npy,0.507859,0.415834
gpt-5.2-chat_run_2.npy,0.508841,0.409995
gpt-5.2-chat_run_3.npy,0.501965,0.403920
standard deviation,0.003037,0.004864
average,0.506221,0.409917
